# T2 prep 2: the model files for the offline run

The offline run can't download the model, so this notebook saves it, once, as its own output:
**`AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP`**, the team's fine-tuned Qwen2.5-VL-7B that
L0 kept on 28 Oct (16.6 GB, public on Hugging Face). Ask Abrar if T2 should use another one.

**Settings:** Internet **on**. Accelerator: None is enough.

**Then:** Save Version → **Save & Run All (Commit)**. Its output folder `model/` becomes an input
of the run notebook (**+ Add Input → Your Work** → this notebook). The output is close to
Kaggle's size limit for one notebook's output (about 20 GB), so this notebook saves nothing else.

In [ ]:
import json, os, shutil, time
from huggingface_hub import HfApi, snapshot_download

REPO = "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP"
OUT = "/kaggle/working/model"

info = HfApi().model_info(REPO, files_metadata=True)
sizes = {s.rfilename: s.size for s in info.siblings}
print("Repo:", REPO, "| commit:", info.sha)
print(f"{len(sizes)} files, {sum(v or 0 for v in sizes.values()) / 1e9:.1f} GB")
print("Free space in /kaggle/working:", round(shutil.disk_usage("/kaggle/working").free / 1e9, 1), "GB")

In [ ]:
t = time.time()
snapshot_download(REPO, revision=info.sha, local_dir=OUT)
shutil.rmtree(os.path.join(OUT, ".cache"), ignore_errors=True)  # only the download's bookkeeping
print(f"Downloaded in {(time.time() - t) / 60:.1f} min")

**Check:** every file is there with Hugging Face's size, every weight shard listed in the index is there, and the chat template is found.

In [ ]:
def bad(name, size):
    path = os.path.join(OUT, name)
    return not os.path.isfile(path) or (size is not None and os.path.getsize(path) != size)

problems = [f"{name}: missing or wrong size" for name, size in sizes.items() if bad(name, size)]
index_path = os.path.join(OUT, "model.safetensors.index.json")
if os.path.isfile(index_path):
    shards = sorted(set(json.load(open(index_path))["weight_map"].values()))
    problems += [f"{s}: shard missing" for s in shards if not os.path.isfile(os.path.join(OUT, s))]

config = json.load(open(os.path.join(OUT, "config.json")))
templates = [n for n in ("chat_template.json", "chat_template.jinja") if os.path.isfile(os.path.join(OUT, n))]
tok_cfg = os.path.join(OUT, "tokenizer_config.json")
if os.path.isfile(tok_cfg) and "chat_template" in json.load(open(tok_cfg)):
    templates.append("tokenizer_config.json")

print("Architecture:", config.get("architectures"), "| dtype:", config.get("torch_dtype") or config.get("dtype"))
print("Chat template in:", templates or "none (the run notebook then uses Qwen2.5-VL's standard template)")
print("Problems:", problems or "none")
assert not problems, "The download is incomplete: run this notebook again."

In [ ]:
source = {"repo": REPO, "commit": info.sha, "downloaded": time.strftime("%Y-%m-%d %H:%M:%S"),
          "chat_template_in": templates, "files": sizes}
with open(os.path.join(OUT, "t2_model_source.json"), "w") as f:
    json.dump(source, f, indent=2)
total = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(OUT) for f in fs)
print(f"Done: {total / 1e9:.1f} GB in {OUT}, and t2_model_source.json")